In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Список файлов
files = {
    'qwen_tqa_cut.csv': 'Qwen_TruthfulQA',
    'qwen_slava_cut.csv': 'Qwen_Slava',
    'yandex_gpt_slava_answered.csv': 'YandexGPT_Slava',
    'yandex_gpt_tqa_answered.csv': 'YandexGPT_TruthfulQA',
    'gpt_slava_annotated.csv': 'ChatGPT_Slava',
    'gpt_tqa_answered.csv': 'ChatGPT_TruthfulQA',
    'giga_slava_annotated.csv': 'GigaChat_Slava',
    'giga_tqa_answered.csv': 'GigaChat_TruthfulQA'
}


feature_metrics = ['Semantic Entropy', 'EigenScore', 'SAR Score', 'NumSets', 'Self Confidence', 'Verbalized Uncertainty']

all_data = []
for file, name in files.items():
    try:
        df = pd.read_csv(file, encoding='utf-8')
        df['source'] = name
        all_data.append(df)
        print(f"✅ Loaded {file}: {len(df)} rows")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

df_all = pd.concat(all_data, ignore_index=True)
print(f"\n📊 Total: {len(df_all)} rows")

available_features = [col for col in feature_metrics if col in df_all.columns]
print(f"✅ Available features: {available_features}")

if 'ground_truth' in df_all.columns:
    df_all['target'] = df_all['ground_truth']
    print(f"Target distribution (overall): {df_all['target'].value_counts().to_dict()}")
else:
    print("❌ ground_truth column not found!")

df_all['model'] = df_all['source'].apply(lambda x: x.split('_')[0])
df_all['dataset'] = df_all['source'].apply(lambda x: x.split('_')[1])

# Выводим статистику по дисбалансу
print("\n" + "="*80)
print("СТАТИСТИКА ДИСБАЛАНСА ПО ДАТАСЕТАМ")
print("="*80)
for name in files.values():
    subset = df_all[df_all['source'] == name]
    true_count = (subset['target'] == 0).sum()
    hal_count = (subset['target'] == 1).sum()
    hal_percent = hal_count / len(subset) * 100
    print(f"{name}: True={true_count}, Hal={hal_count}, Hal%={hal_percent:.1f}%")

✅ Loaded qwen_tqa_cut.csv: 790 rows
✅ Loaded qwen_slava_cut.csv: 500 rows
✅ Loaded yandex_gpt_slava_answered.csv: 481 rows
✅ Loaded yandex_gpt_tqa_answered.csv: 755 rows
✅ Loaded gpt_slava_annotated.csv: 500 rows
✅ Loaded gpt_tqa_answered.csv: 785 rows
✅ Loaded giga_slava_annotated.csv: 500 rows
✅ Loaded giga_tqa_answered.csv: 761 rows

📊 Total: 5072 rows
✅ Available features: ['Semantic Entropy', 'EigenScore', 'SAR Score', 'NumSets', 'Self Confidence', 'Verbalized Uncertainty']
Target distribution (overall): {0: 4014, 1: 1058}

СТАТИСТИКА ДИСБАЛАНСА ПО ДАТАСЕТАМ
Qwen_TruthfulQA: True=452, Hal=338, Hal%=42.8%
Qwen_Slava: True=241, Hal=259, Hal%=51.8%
YandexGPT_Slava: True=439, Hal=42, Hal%=8.7%
YandexGPT_TruthfulQA: True=654, Hal=101, Hal%=13.4%
ChatGPT_Slava: True=460, Hal=40, Hal%=8.0%
ChatGPT_TruthfulQA: True=681, Hal=104, Hal%=13.2%
GigaChat_Slava: True=457, Hal=43, Hal%=8.6%
GigaChat_TruthfulQA: True=630, Hal=131, Hal%=17.2%


In [ ]:
def downsample_dataset(df, target_col='target', random_state=42):

    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]

    if len(df_minority) >= len(df_majority):
        print(f"    Классы уже сбалансированы или миноритарный больше: маж= {len(df_majority)}, мин={len(df_minority)}")
        return df

    df_majority_downsampled = resample(
        df_majority,
        replace=False,
        n_samples=len(df_minority),
        random_state=random_state
    )

    df_balanced = pd.concat([df_majority_downsampled, df_minority])

    print(f"    Даунсемплинг: {len(df_majority)} → {len(df_majority_downsampled)} (мажоритарный), миноритарный: {len(df_minority)}")
    return df_balanced

In [ ]:

# МОДЕЛИ И ГИПЕРПАРАМЕТРЫ


# 1. Logistic Regression
logreg_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'class_weight': ['balanced', None]
}

# 2. Random Forest
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

# 3. Gradient Boosting
gb_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

# 4. SVM
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

# 5. MLP (Neural Network)
mlp_params = {
    'hidden_layer_sizes': [(25,), (50,), (50, 25)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive']
}

# 6. XGBoost
xgb_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'eval_metric': ['logloss']
}

# 7. CatBoost
catboost_params = {
    'iterations': [50, 100],
    'depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [32, 64]
}


models_with_params = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), logreg_params),
    'Random Forest': (RandomForestClassifier(random_state=42), rf_params),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=42), gb_params),
    'SVM': (SVC(probability=True, random_state=42), svm_params),
    'MLP': (MLPClassifier(max_iter=500, random_state=42), mlp_params),
    'XGBoost': (XGBClassifier(random_state=42, use_label_encoder=False, verbosity=0), xgb_params),
    'CatBoost': (CatBoostClassifier(random_state=42, verbose=0), catboost_params)
}

In [ ]:

#  ОЦЕНКА С ДАУНСЕМПЛИНГОМ(recall opti)

def evaluate_with_downsampling(df, dataset_name, features):

    X = df[features].fillna(df[features].mean())
    y = df['target']

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    results = []
    for name, (model, params) in models_with_params.items():
        print(f"  Optimizing {name} on {dataset_name}...")

        # GridSearchCV для подбора гиперпараметров
        grid_search = GridSearchCV(
            model, params,
            cv=3,
            scoring='recall',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X, y)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        f1_scores = []
        precision_scores = []
        recall_scores = []
        roc_scores = []

        for train_idx, test_idx in cv.split(X, y):
            X_train = X.iloc[train_idx]
            y_train = y.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_test = y.iloc[test_idx]


            train_df = pd.DataFrame(X_train)
            train_df['target'] = y_train.values


            train_df_balanced = downsample_dataset(train_df, target_col='target')

            X_train_balanced = train_df_balanced[features]
            y_train_balanced = train_df_balanced['target']

            model_clone = best_model.__class__(**best_model.get_params())
            model_clone.fit(X_train_balanced, y_train_balanced)

            if hasattr(model_clone, "predict_proba"):
                y_proba = model_clone.predict_proba(X_test)[:, 1]
                y_pred = (y_proba > 0.5).astype(int)
                roc_scores.append(roc_auc_score(y_test, y_proba))
            else:
                y_pred = model_clone.predict(X_test)
                roc_scores.append(0.5)

            f1_scores.append(f1_score(y_test, y_pred, zero_division=0))
            precision_scores.append(precision_score(y_test, y_pred, zero_division=0))
            recall_scores.append(recall_score(y_test, y_pred, zero_division=0))

        results.append({
            'dataset': dataset_name,
            'model': name,
            'best_params': str(best_params),
            'f1': np.mean(f1_scores),
            'f1_std': np.std(f1_scores),
            'precision': np.mean(precision_scores),
            'precision_std': np.std(precision_scores),
            'recall': np.mean(recall_scores),
            'recall_std': np.std(recall_scores),
            'roc_auc': np.mean(roc_scores),
            'roc_auc_std': np.std(roc_scores)
        })

    return pd.DataFrame(results)

In [ ]:

#  ЗАПУСК НА ВСЕХ СРЕЗАХ

all_results = []

print("="*80)
print("1. ОТДЕЛЬНЫЕ ФАЙЛЫ (8 шт) - С ДАУНСЕМПЛИНГОМ")
print("="*80)

for file, name in files.items():
    try:
        df = pd.read_csv(file, encoding='utf-8')
        if 'ground_truth' in df.columns:
            df['target'] = df['ground_truth']
        print(f"\n📁 Processing {name} ({len(df)} samples)...")
        results = evaluate_with_downsampling(df, name, available_features)
        all_results.append(results)
        print(f"✅ {name} completed")
    except Exception as e:
        print(f"❌ Error on {file}: {e}")

print("\n" + "="*80)
print("2. ПО ДАТАСЕТАМ (TruthfulQA vs Slava) - С ДАУНСЕМПЛИНГОМ")
print("="*80)

for dataset_name in ['TruthfulQA', 'Slava']:
    df_subset = df_all[df_all['dataset'] == dataset_name]
    if len(df_subset) > 0:
        print(f"\n📁 Processing ALL_{dataset_name} ({len(df_subset)} samples)...")
        results = evaluate_with_downsampling(df_subset, f"ALL_{dataset_name}", available_features)
        all_results.append(results)
        print(f"✅ ALL_{dataset_name} completed")

print("\n" + "="*80)
print("3. ПО МОДЕЛЯМ - С ДАУНСЕМПЛИНГОМ")
print("="*80)

for model_name in ['Qwen', 'YandexGPT', 'ChatGPT', 'GigaChat']:
    df_subset = df_all[df_all['model'] == model_name]
    if len(df_subset) > 0:
        print(f"\n📁 Processing ALL_{model_name} ({len(df_subset)} samples)...")
        results = evaluate_with_downsampling(df_subset, f"ALL_{model_name}", available_features)
        all_results.append(results)
        print(f"✅ ALL_{model_name} completed")

print("\n" + "="*80)
print("4. ПОЛНЫЙ ДАТАСЕТ - С ДАУНСЕМПЛИНГОМ")
print("="*80)

print(f"\n📁 Processing FULL_DATASET ({len(df_all)} samples)...")
results = evaluate_with_downsampling(df_all, "FULL_DATASET", available_features)
all_results.append(results)
print("✅ FULL_DATASET completed")

# Объединение
df_all_results = pd.concat(all_results, ignore_index=True)
print(f"\n📊 Total evaluations: {len(df_all_results)}")

1. ОТДЕЛЬНЫЕ ФАЙЛЫ (8 шт) - С ДАУНСЕМПЛИНГОМ

📁 Processing Qwen_TruthfulQA (790 samples)...
  Optimizing Logistic Regression on Qwen_TruthfulQA...
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
  Optimizing Random Forest on Qwen_TruthfulQA...
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
    Даунсемплинг: 362 → 270 (мажоритарный), миноритарный: 270
  Optimizing Gradient Boosting on Qwen_TruthfulQA...
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Даунсемплинг: 361 → 271 (мажоритарный), миноритарный: 271
    Дау

In [ ]:
#Results

print("="*80)
print("ТАБЛИЦЫ ПО КАЖДОМУ СРЕЗУ (С ДАУНСЕМПЛИНГОМ)")
print("="*80)

def save_table(df, slice_names, filename, title):
    df_subset = df[df['dataset'].isin(slice_names)]
    if df_subset.empty:
        return None
    table = df_subset[['dataset', 'model', 'recall', 'f1', 'precision', 'roc_auc']].copy()
    table = table.round(4)
    table = table.sort_values(['dataset', 'recall'], ascending=[True, False])
    print(f"\n{title}")
    display(table)
    table.to_csv(filename, index=False)
    return table

# Таблица 1: Отдельные файлы
individual_names = list(files.values())
table1 = save_table(df_all_results, individual_names, 'table1_individual_downsampled.csv', "📊 ТАБЛИЦА 1: ОТДЕЛЬНЫЕ ФАЙЛЫ (downsampled)")

# Таблица 2: По датасетам
dataset_names = [d for d in df_all_results['dataset'].unique() if d.startswith('ALL_TruthfulQA') or d.startswith('ALL_Slava')]
table2 = save_table(df_all_results, dataset_names, 'table2_by_dataset_downsampled.csv', "📊 ТАБЛИЦА 2: ПО ДАТАСЕТАМ (downsampled)")

# Таблица 3: По моделям
model_names = [d for d in df_all_results['dataset'].unique() if d.startswith('ALL_Qwen') or d.startswith('ALL_YandexGPT') or d.startswith('ALL_ChatGPT') or d.startswith('ALL_GigaChat')]
table3 = save_table(df_all_results, model_names, 'table3_by_model_downsampled.csv', "📊 ТАБЛИЦА 3: ПО МОДЕЛЯМ (downsampled)")

# Таблица 4: Полный датасет
table4 = save_table(df_all_results, ['FULL_DATASET'], 'table4_full_downsampled.csv', "📊 ТАБЛИЦА 4: ПОЛНЫЙ ДАТАСЕТ (downsampled)")

ТАБЛИЦЫ ПО КАЖДОМУ СРЕЗУ (С ДАУНСЕМПЛИНГОМ)

📊 ТАБЛИЦА 1: ОТДЕЛЬНЫЕ ФАЙЛЫ (downsampled)


,dataset,model,recall,f1,precision,roc_auc
29,ChatGPT_Slava,Random Forest,0.7000,0.2751,0.1726,0.7332
30,ChatGPT_Slava,Gradient Boosting,0.6000,0.2404,0.1518,0.6492
32,ChatGPT_Slava,MLP,0.6000,0.2794,0.1830,0.7477
33,ChatGPT_Slava,XGBoost,0.6000,0.2554,0.1641,0.6863
34,ChatGPT_Slava,CatBoost,0.5750,0.2437,0.1565,0.6895
28,ChatGPT_Slava,Logistic Regression,0.5250,0.2778,0.1900,0.7463
31,ChatGPT_Slava,SVM,0.5250,0.2980,0.2089,0.8004
41,ChatGPT_TruthfulQA,CatBoost,0.6924,0.4263,0.3085,0.8062
36,ChatGPT_TruthfulQA,Random Forest,0.6829,0.3945,0.2778,0.7817
39,ChatGPT_TruthfulQA,MLP,0.6648,0.4160,0.3113,0.7991



📊 ТАБЛИЦА 2: ПО ДАТАСЕТАМ (downsampled)


,dataset,model,recall,f1,precision,roc_auc
67,ALL_Slava,MLP,0.7553,0.5412,0.4224,0.7894
66,ALL_Slava,SVM,0.7371,0.5459,0.4341,0.7950
69,ALL_Slava,CatBoost,0.7319,0.5421,0.4321,0.8081
65,ALL_Slava,Gradient Boosting,0.7265,0.4953,0.3761,0.7754
64,ALL_Slava,Random Forest,0.7214,0.5177,0.4047,0.7905
63,ALL_Slava,Logistic Regression,0.7137,0.5551,0.4555,0.7858
68,ALL_Slava,XGBoost,0.6953,0.4947,0.3849,0.7771
62,ALL_TruthfulQA,CatBoost,0.6840,0.4774,0.3668,0.7336
58,ALL_TruthfulQA,Gradient Boosting,0.6766,0.4666,0.3565,0.7147
57,ALL_TruthfulQA,Random Forest,0.6677,0.4692,0.3618,0.7202



📊 ТАБЛИЦА 3: ПО МОДЕЛЯМ (downsampled)


,dataset,model,recall,f1,precision,roc_auc
85,ALL_ChatGPT,Random Forest,0.6805,0.3197,0.2095,0.7440
90,ALL_ChatGPT,CatBoost,0.6596,0.3604,0.2488,0.7492
87,ALL_ChatGPT,SVM,0.6318,0.3907,0.2833,0.7715
89,ALL_ChatGPT,XGBoost,0.6185,0.2941,0.1936,0.7153
86,ALL_ChatGPT,Gradient Boosting,0.6113,0.2944,0.1943,0.7071
88,ALL_ChatGPT,MLP,0.5968,0.3948,0.2960,0.7460
84,ALL_ChatGPT,Logistic Regression,0.5899,0.3922,0.2940,0.7462
92,ALL_GigaChat,Random Forest,0.5919,0.2556,0.1630,0.5498
91,ALL_GigaChat,Logistic Regression,0.5459,0.2693,0.1797,0.5821
96,ALL_GigaChat,XGBoost,0.5339,0.2242,0.1422,0.5232



📊 ТАБЛИЦА 4: ПОЛНЫЙ ДАТАСЕТ (downsampled)


,dataset,model,recall,f1,precision,roc_auc
104,FULL_DATASET,CatBoost,0.7080,0.4896,0.3747,0.7504
98,FULL_DATASET,Logistic Regression,0.6910,0.4831,0.3717,0.7277
99,FULL_DATASET,Random Forest,0.6844,0.4800,0.3700,0.7393
102,FULL_DATASET,MLP,0.6778,0.4828,0.3758,0.7348
100,FULL_DATASET,Gradient Boosting,0.6693,0.4678,0.3601,0.7306
103,FULL_DATASET,XGBoost,0.6570,0.4666,0.3622,0.7264
101,FULL_DATASET,SVM,0.6210,0.4723,0.3815,0.7292
